**ResNet3-18 Model Inference Code Final**

In [1]:
# 의존성 정의
import os
import torch
import cv2
import numpy as np
from torchvision.models.video import r3d_18, R3D_18_Weights
from torch import nn
from torchvision import transforms

In [2]:
# GPU 사용 확인, else CPU
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Device being used:", device)

Device being used: cuda:0


In [3]:
# 클래스 라벨 정의
class_names = ['real', 'fake']

In [4]:
# 모델 로딩
def load_model(checkpoint_path):
    
    print("Loading model from:", checkpoint_path)
    model = r3d_18(weights=None)
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, 2)

    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['state_dict'])
    model.to(device)
    model.eval()
    
    return model

In [5]:
# 비디오 전처리
def preprocess_video(video_path, clip_len=16, resize_height=256, resize_width=256, crop_size=224, augmentation=True):
    
    cap = cv2.VideoCapture(video_path)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if frame_count < clip_len:
        raise ValueError(f"프레임 수가 너무 적습니다: {frame_count}")

    EXTRACT_FREQUENCY = 4
    while EXTRACT_FREQUENCY > 1 and frame_count // EXTRACT_FREQUENCY <= clip_len:
        EXTRACT_FREQUENCY -= 1

    frames = []
    count = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if count % EXTRACT_FREQUENCY == 0:
            frame = cv2.resize(frame, (resize_width, resize_height))
            frames.append(frame)

        count += 1

    cap.release()

    if len(frames) < clip_len:
        frames += [frames[-1]] * (clip_len - len(frames))

    # 무작위 시작 인덱스
    start_index = np.random.randint(0, len(frames) - clip_len + 1)
    selected_frames = frames[start_index: start_index + clip_len]

    # 무작위 Crop
    height_index = np.random.randint(resize_height - crop_size)
    width_index = np.random.randint(resize_width - crop_size)

    cropped_frames = []
    for frame in selected_frames:
        frame = frame[height_index:height_index+crop_size,
                      width_index:width_index+crop_size]

        if augmentation and np.random.random() < 0.5:
            frame = cv2.flip(frame, flipCode=1)

        frame = frame[:, :, [2, 1, 0]]  # BGR → RGB
        cropped_frames.append(frame)

    # 정규화
    cropped_frames = [(frame.astype(np.float32) - np.array([[[90.0, 98.0, 102.0]]], dtype=np.float32)) for frame in cropped_frames]

    # 텐서 변환
    buffer = np.stack(cropped_frames, axis=0)   # (clip_len, H, W, C)
    buffer = buffer.transpose((3, 0, 1, 2))     # (C, clip_len, H, W)

    return torch.from_numpy(buffer).unsqueeze(0)

In [6]:
# 추론
def run_inference(model, video_tensor):
    
    with torch.no_grad():
        video_tensor = video_tensor.to(device)
        outputs = model(video_tensor)
        probs = nn.Softmax(dim=1)(outputs)
        preds = torch.argmax(probs, dim=1)

    for i, pred in enumerate(preds):
        print(f"Clip {i+1}: Prediction: {class_names[pred]} (Confidence: {probs[i][pred]:.4f})")

In [7]:
# 실행
if __name__ == '__main__':
    
    video_path = "id0_0006.mp4"
    checkpoint_path = "run/run_0/models/R3D_model-celeb-df_epoch-2.pth.tar"

    model = load_model(checkpoint_path)
    video_tensor = preprocess_video(video_path)
    run_inference(model, video_tensor)

Loading model from: run/run_0/models/R3D_model-celeb-df_epoch-2.pth.tar


C:\Users\user\AppData\Local\Temp\ipykernel_26252\3784470510.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location=device)

Clip 1: Prediction: fake (Confidence: 0.5858)
